# ANN
A two-layer neural network for multi-output regression, derived using matrix calculus.

Let $N$ be the batch size, $d$ the number of input features, $H$ the number of hidden units, and $K$ the number of regression outputs.

$$
\begin{align*}
\mathbf{Z} &= \mathbf{X}\mathbf{W}_1 + \mathbf{1}_N\mathbf{b}_1, \\
\mathbf{A} &= \operatorname{ReLU}(\mathbf{Z}), \\
\widehat{\mathbf{Y}} &= \mathbf{A}\mathbf{W}_2 + \mathbf{1}_N\mathbf{b}_2.
\end{align*}
$$

Shapes:
- $\mathbf{X} \in \mathbb{R}^{N \times d}$ and $\mathbf{Y} \in \mathbb{R}^{N \times K}$
- $\mathbf{Z},\mathbf{A} \in \mathbb{R}^{N \times H}$ and $\widehat{\mathbf{Y}} \in \mathbb{R}^{N \times K}$
- $\mathbf{W}_1 \in \mathbb{R}^{d \times H}$ and $\mathbf{b}_1 \in \mathbb{R}^{1 \times H}$
- $\mathbf{W}_2 \in \mathbb{R}^{H \times K}$ and $\mathbf{b}_2 \in \mathbb{R}^{1 \times K}$

$\mathbf{1}_N \in \mathbb{R}^{N \times 1}$ represents row-wise bias broadcasting.

## Forward Pass

$$
\mathbf{Z}=\mathbf{X}\mathbf{W}_1+\mathbf{1}_N\mathbf{b}_1,
\qquad
\mathbf{A}=\operatorname{ReLU}(\mathbf{Z}),
\qquad
\widehat{\mathbf{Y}}=\mathbf{A}\mathbf{W}_2+\mathbf{1}_N\mathbf{b}_2.
$$

Each row of $\widehat{\mathbf{Y}}$ contains $K$ continuous predictions. The output layer is linear.

## Backward Pass with Matrix Differentials

For mean squared error averaged over all $NK$ prediction elements, define $\mathbf{E}=\widehat{\mathbf{Y}}-\mathbf{Y}$:

$$
L=\frac{1}{NK}\lVert\mathbf{E}\rVert_F^2
=\frac{1}{NK}\operatorname{tr}(\mathbf{E}^T\mathbf{E}).
$$

Using $d\operatorname{tr}(\mathbf{E}^T\mathbf{E})=2\operatorname{tr}(\mathbf{E}^T d\mathbf{E})$,

$$
dL=\operatorname{tr}(\mathbf{G}^T d\widehat{\mathbf{Y}}),
\qquad
\mathbf{G}:=\nabla_{\widehat{\mathbf{Y}}}L
=\frac{2}{NK}(\widehat{\mathbf{Y}}-\mathbf{Y})
\in\mathbb{R}^{N\times K}.
$$

For the output layer,

$$
d\widehat{\mathbf{Y}}
=d\mathbf{A}\,\mathbf{W}_2
+\mathbf{A}\,d\mathbf{W}_2
+\mathbf{1}_N\,d\mathbf{b}_2.
$$

Substituting into $dL$ and using the cyclic trace identity gives

$$
\boxed{
\begin{aligned}
\nabla_{\mathbf{W}_2}L &= \mathbf{A}^T\mathbf{G} && (H\times K),\\
\nabla_{\mathbf{b}_2}L &= \mathbf{1}_N^T\mathbf{G} && (1\times K),\\
\nabla_{\mathbf{A}}L &= \mathbf{G}\mathbf{W}_2^T && (N\times H).
\end{aligned}}
$$

For ReLU, let $\mathbf{M}=\mathbb{1}[\mathbf{Z}>0]$. Then

$$
\mathbf{D}:=\nabla_{\mathbf{Z}}L
=(\mathbf{G}\mathbf{W}_2^T)\odot\mathbf{M}
\in\mathbb{R}^{N\times H}.
$$

Since $d\mathbf{Z}=\mathbf{X}\,d\mathbf{W}_1+\mathbf{1}_N\,d\mathbf{b}_1$,

$$
\boxed{
\begin{aligned}
\nabla_{\mathbf{W}_1}L &= \mathbf{X}^T\mathbf{D} && (d\times H),\\
\nabla_{\mathbf{b}_1}L &= \mathbf{1}_N^T\mathbf{D} && (1\times H).
\end{aligned}}
$$

When $K=1$, these equations reduce to the scalar-output case. For $K>1$, the important difference is that `np.mean` averages over both $N$ and $K$, so the output gradient contains the factor $1/(NK)$.

## Implementation using NumPy

In [1]:
import numpy as np


class NeuralNetwork:
    def __init__(self, in_features, hidden_units, out_features):
        self.W1 = np.random.randn(in_features, hidden_units) * 0.01
        self.b1 = np.zeros((hidden_units,))
        self.W2 = np.random.randn(hidden_units, out_features) * 0.01
        self.b2 = np.zeros((out_features,))

    def forward(self, x):
        h = x @ self.W1 + self.b1
        a = np.maximum(0, h)
        out = a @ self.W2 + self.b2
        return out, (h, a)

    def backward(self, x, h, a, G):
        """Compute gradients given G = dL/dout with shape (N, K)."""
        dW2 = a.T @ G
        db2 = G.sum(axis=0)

        dA = G @ self.W2.T
        dZ = dA * (h > 0)

        dW1 = x.T @ dZ
        db1 = dZ.sum(axis=0)
        return dW1, db1, dW2, db2

In [2]:
# Synthetic multi-output regression data
rng = np.random.default_rng(0)
np.random.seed(0)

samples = 500
in_features = 20
hidden_units = 10
out_features = 3

X = rng.normal(size=(samples, in_features))
true_W = rng.normal(size=(in_features, out_features))
true_b = rng.normal(size=(out_features,))
y = X @ true_W + true_b + 0.1 * rng.normal(size=(samples, out_features))

max_iters = 2000
learning_rate = 0.02
nn = NeuralNetwork(in_features, hidden_units, out_features)

for epoch in range(max_iters):
    out, (h, a) = nn.forward(X)

    error = out - y
    loss = np.mean(error ** 2)
    G = (2 / error.size) * error  # error.size = N * K

    dW1, db1, dW2, db2 = nn.backward(X, h, a, G)

    nn.W1 -= learning_rate * dW1
    nn.b1 -= learning_rate * db1
    nn.W2 -= learning_rate * dW2
    nn.b2 -= learning_rate * db2

    if epoch % 200 == 0:
        print(f"Epoch {epoch:4d}, Loss: {loss:.4f}")

print("prediction shape:", nn.forward(X)[0].shape)

Epoch    0, Loss: 19.9076
Epoch  200, Loss: 1.1639
Epoch  400, Loss: 0.2352
Epoch  600, Loss: 0.1472
Epoch  800, Loss: 0.1091
Epoch 1000, Loss: 0.0851
Epoch 1200, Loss: 0.0699
Epoch 1400, Loss: 0.0590
Epoch 1600, Loss: 0.0504
Epoch 1800, Loss: 0.0421
prediction shape: (500, 3)
